# GAT with SAGPool Session Readout

This notebook follows the SR-GNN / TAGNN session recommendation pipeline, but replaces the GGNN encoder with a Graph Attention Network and the readout with a SAGPool-style hierarchical pooling block.

After a clean run, `RESULTS_DIR` contains:

- `gat_sagpool_v2_digi_reg_results.csv`, `gat_sagpool_v2_digi_reg_history.csv`, … (Run 2; see `RESULT_FILE_PREFIX` in config cell)
- `checkpoints/gat_sagpool_v2_digi_reg_<dataset>.pt` (best-by-`MRR@20`, plus final test metrics in the same file)
- updated `model_summary.csv` and `gat_three_way_comparison.csv` when prior baselines are present

## Architecture (target end state)

A session prefix `[v1, v2, ..., vt]` is turned into a graph. Nodes are unique items from the prefix. Directed edges follow observed clicks, for example `v1 -> v2`; reverse edges are also built so the model can read both incoming and outgoing transition context. Repeated transitions are normalized and passed as edge weights.

The encoder is a bidirectional GAT shared with the other two notebooks (same `BidirectionalGATEncoder` interface). The readout is replaced by SAGPool-style hierarchical pooling over the session graph:

```text
session prefix
     |
directed session graph
     |
item embedding, dim=100
     |
     +--> forward GATConv: 4 heads x 25 dims, dropout=0.1, edge weights
     |
     +--> backward GATConv: 4 heads x 25 dims, dropout=0.1, edge weights
                 |
concat forward/backward states, dim=200
                 |
linear direction fusion, 200 -> 100
                 |
contextual node embeddings
                 |
SAGPool readout : score nodes, keep top-k, aggregate
                 |
session representation
                 |
scores for all items
```

GAT details kept identical to the other two notebooks (so encoder differences are isolated only by the readout):

- Layer type: `torch_geometric.nn.GATConv`.
- Hidden size: `100`.
- Number of GAT layers per direction: `1`.
- Attention heads: `4`.
- Per-head output size: `25`, concatenated back to `100`.
- Dropout inside GAT and after activation: `0.1`.
- Activation: `ELU`.
- Residual connection: enabled inside each directional stack.
- Self-loops: enabled by `GATConv`.
- Edge features: normalized transition weights passed as one-dimensional edge attributes.
- Direction fusion: concatenate forward and backward node states, then apply a linear layer `200 -> 100`.

## Prediction (target end state)

The final session vector is multiplied by the item embedding matrix. This gives one score for every candidate item. Items are ranked by score, and the top 20 are used for `Precision@20` and `MRR@20`.

This notebook prepares raw Yoochoose and Diginetica inputs, runs the SR-GNN / TAGNN preprocessing pipeline, and is wired so that adding the SAGPool model requires only:

1. defining `GATSAGPool` class
2. providing a factory `lambda num_items, config: GATSAGPool(...)` to `run_dataset_experiment`

Expected Kaggle input directories:
- `/kaggle/input/datasets/chadgostopp/recsys-challenge-2015` containing `yoochoose-clicks.dat`
- `/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset` containing `train-item-views.csv`

`Yoochoose 1/4` is skipped because it is too large for the target memory budget.

## Environment and Dependencies

Kaggle base image already provides `torch`, `pandas`, `numpy`, `matplotlib`, and `kagglehub`. The PyG stack (`torch_geometric`, used by GATConv / SAGPooling / GraphConv / `global_*_pool`) is not preinstalled, so it is installed here.

The install cell is idempotent (skips if `torch_geometric` is already importable) and quiet. From PyG `>=2.3` the base `pip install torch_geometric` is enough for everything this notebook uses; `pyg_lib`, `torch_scatter`, `torch_sparse` are not required.

Kaggle settings required for a clean first run:

- **Accelerator: GPU** (T4 is sufficient).
- **Internet: On** (otherwise the pip install fails).
- **Environment: latest** (Settings -> Environment -> "Always use latest environment") - avoids torch / torch_geometric version mismatches that occasionally appear with pinned-old environments.

After the install, the cell prints the resolved `torch` and `torch_geometric` versions plus `cuda_available`, so any environment problem is visible at the very top of the run instead of mid-training.

In [1]:
import importlib
import importlib.util
import subprocess
import sys


def _ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package],
            check=True,
        )
        importlib.invalidate_caches()


_ensure_package("torch_geometric")

import torch  # noqa: E402
import torch_geometric  # noqa: E402

print(f"torch={torch.__version__}")
print(f"torch_geometric={torch_geometric.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.2 MB/s eta 0:00:00
torch=2.10.0+cu128
torch_geometric=2.7.0
cuda_available=True


## Prepare Datasets


In [2]:
import math
import os
import random
import time
from collections import Counter
from datetime import date
from pathlib import Path

MPLCONFIGDIR = Path("/tmp/matplotlib")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import (
    GATConv,
    GraphConv,
    SAGPooling,
    global_max_pool,
    global_mean_pool,
)
from torch_geometric.utils import softmax as pyg_softmax

In [3]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

YOOCHOOSE_INPUT_DIR = KAGGLE_INPUT_DIR / "datasets/chadgostopp/recsys-challenge-2015"
DIGINETICA_INPUT_DIR = (
    KAGGLE_INPUT_DIR / "datasets/profalbusdumbledore/diginetica-dataset"
)
YOOCHOOSE_SOURCE = YOOCHOOSE_INPUT_DIR / "yoochoose-clicks.dat"
DIGINETICA_SOURCE = DIGINETICA_INPUT_DIR / "train-item-views.csv"

RESULTS_DIR = OUTPUT_DIR / "results"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [RESULTS_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"output_dir={OUTPUT_DIR}")
print(f"Using Yoochoose source: {YOOCHOOSE_SOURCE}")
print(f"Using Diginetica source: {DIGINETICA_SOURCE}")

output_dir=/kaggle/working
Using Yoochoose source: /kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-clicks.dat
Using Diginetica source: /kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-item-views.csv


## Preprocess Sessions

The preprocessing flow builds ordered sessions, removes short sessions and rare items, splits chronologically, remaps item ids from training data, and expands sessions into prefix-label examples.

In [4]:
def load_yoochoose_sessions(path):
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["session_id", "timestamp", "item_id"],
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)

    session_items = {}
    session_dates = {}
    current_session_id = None
    current_timestamp = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_timestamp is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_timestamp

        current_session_id = session_id
        current_timestamp = row.timestamp

        if session_id in session_items:
            session_items[session_id].append(row.item_id)
        else:
            session_items[session_id] = [row.item_id]

    if current_session_id is not None:
        session_dates[current_session_id] = current_timestamp

    return [
        (session_id, session_dates[session_id], items)
        for session_id, items in session_items.items()
    ]


def load_diginetica_sessions(path):
    df = pd.read_csv(
        path,
        sep=";",
        usecols=["sessionId", "itemId", "timeframe", "eventdate"],
    )
    df = df.rename(columns={"sessionId": "session_id", "itemId": "item_id"})
    df["eventdate"] = pd.to_datetime(df["eventdate"], format="%Y-%m-%d")

    session_clicks = {}
    session_dates = {}
    current_session_id = None
    current_date = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_date is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_date

        current_session_id = session_id
        current_date = row.eventdate

        click = (row.item_id, int(row.timeframe))
        if session_id in session_clicks:
            session_clicks[session_id].append(click)
        else:
            session_clicks[session_id] = [click]

    if current_session_id is not None:
        session_dates[current_session_id] = current_date

    sessions = []
    for session_id, clicks in session_clicks.items():
        ordered_clicks = sorted(clicks, key=lambda click: click[1])
        items = [item for item, _ in ordered_clicks]
        sessions.append((session_id, session_dates[session_id], items))
    return sessions

In [5]:
def drop_short_sessions(sessions):
    return [session for session in sessions if len(session[2]) >= 2]


def drop_rare_items(sessions, min_freq=5):
    counts = Counter()
    for _, _, items in sessions:
        counts.update(items)

    result = []
    for session_id, date, items in sessions:
        kept = [i for i in items if counts[i] >= min_freq]
        if len(kept) >= 2:
            result.append((session_id, date, kept))
    return result


def sort_by_date(sessions):
    return sorted(sessions, key=lambda session: session[1])


def split_by_date(sessions, test_days):
    max_date = max(date for _, date, _ in sessions)
    split_date = max_date - pd.Timedelta(days=test_days)
    train = [s for s in sessions if s[1] < split_date]
    test = [s for s in sessions if s[1] > split_date]
    return train, test


def renumber_training_items(train_sessions):
    item_to_index = {}
    next_item_index = 1
    remapped_sessions = []

    for session_id, date, items in train_sessions:
        remapped_items = []
        for item in items:
            if item not in item_to_index:
                item_to_index[item] = next_item_index
                next_item_index += 1
            remapped_items.append(item_to_index[item])
        remapped_sessions.append((session_id, date, remapped_items))

    return remapped_sessions, item_to_index


def remap_test_sessions(test_sessions, item_to_index):
    remapped_sessions = []
    for session_id, date, items in test_sessions:
        remapped_items = [
            item_to_index[item] for item in items if item in item_to_index
        ]
        if len(remapped_items) >= 2:
            remapped_sessions.append((session_id, date, remapped_items))
    return remapped_sessions


def expand_sessions(sessions):
    examples = []
    for session_id, date, items in sessions:
        for reverse_offset in range(1, len(items)):
            examples.append(
                (session_id, date, items[:-reverse_offset], items[-reverse_offset])
            )
    return examples


def keep_recent_fraction(examples, denominator):
    if denominator is None:
        return examples
    keep = len(examples) // denominator
    return examples[-keep:] if keep else examples


def prefix_label_rows(examples):
    return [(list(prefix), int(label)) for _, _, prefix, label in examples]


def vocabulary_size_from_rows(*row_groups):
    max_item_id = 0
    for rows in row_groups:
        for prefix, label in rows:
            max_item_id = max(max_item_id, int(label), max(prefix))
    return max_item_id + 1


def preprocess(sessions, test_days, train_fraction_denominator):
    sessions = drop_short_sessions(sessions)
    sessions = drop_rare_items(sessions)
    sessions = sort_by_date(sessions)
    train_sessions, test_sessions = split_by_date(sessions, test_days)

    train_sessions, item_to_index = renumber_training_items(train_sessions)
    test_sessions = remap_test_sessions(test_sessions, item_to_index)

    train_examples = expand_sessions(train_sessions)
    test_examples = expand_sessions(test_sessions)
    train_examples = keep_recent_fraction(train_examples, train_fraction_denominator)
    return train_examples, test_examples

In [6]:
yoochoose_sessions = load_yoochoose_sessions(YOOCHOOSE_SOURCE)
diginetica_sessions = load_diginetica_sessions(DIGINETICA_SOURCE)

yoochoose_1_64_train, yoochoose_1_64_test = preprocess(
    yoochoose_sessions, test_days=1, train_fraction_denominator=64
)
diginetica_train, diginetica_test = preprocess(
    diginetica_sessions, test_days=7, train_fraction_denominator=None
)

YOOCHOOSE_1_64_TRAIN_ROWS = prefix_label_rows(yoochoose_1_64_train)
YOOCHOOSE_1_64_TEST_ROWS = prefix_label_rows(yoochoose_1_64_test)
DIGINETICA_TRAIN_ROWS = prefix_label_rows(diginetica_train)
DIGINETICA_TEST_ROWS = prefix_label_rows(diginetica_test)

print(f"Yoochoose 1/64 train examples: {len(YOOCHOOSE_1_64_TRAIN_ROWS):,}")
print(f"Yoochoose 1/64 test examples: {len(YOOCHOOSE_1_64_TEST_ROWS):,}")
print(f"Diginetica train examples: {len(DIGINETICA_TRAIN_ROWS):,}")
print(f"Diginetica test examples: {len(DIGINETICA_TEST_ROWS):,}")

Yoochoose 1/64 train examples: 369,859
Yoochoose 1/64 test examples: 55,898
Diginetica train examples: 719,470
Diginetica test examples: 60,858


In [7]:
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

if torch.cuda.is_available():
    print(f"cuda_device={torch.cuda.get_device_name(0)}")
    print(
        f"cuda_memory_gb={torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}"
    )
else:
    print("cuda_device=None")

print("Available Kaggle input files:")
for dirname, _, filenames in os.walk(KAGGLE_INPUT_DIR):
    for filename in filenames:
        print(Path(dirname) / filename)

cuda_device=Tesla T4
cuda_memory_gb=14.6
Available Kaggle input files:
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/products.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/product-categories.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-queries.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-item-views.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-clicks.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-purchases.csv
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-buys.dat
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-clicks.dat
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-test.dat
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/dataset-README.txt
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-data/yoochoose-buys.dat
/kaggle/input/datasets/chadgostopp/recsys-challe

## Data loading and graph construction helpers

The session-graph contract is intentionally identical to `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`. This guarantees that the only thing changing across the three GAT branches is the readout (and, by extension, the part of the model after the encoder), so any difference in `Precision@20` / `MRR@20` is attributable to the readout, not to graph construction.

Each session prefix becomes a `torch_geometric.data.Data` object with:

- `x`: original training-vocabulary item IDs for the unique nodes in the prefix; this is the input to the shared item embedding table.
- `edge_index` / `edge_weight`: forward (click order) edges with normalized transition weights (default channel used by PyG mini-batch utilities).
- `forward_edge_index` / `forward_edge_weight`: forward transition edges with normalized weights.
- `backward_edge_index` / `backward_edge_weight`: reverse transition edges with normalized weights.
- `sequence`: per-position local node indices, so contextual node embeddings can be re-aligned with the original click order at readout time.
- `sequence_length`: length of the original prefix (used during batched scatter / pooling operations).
- `last_click`: local node index of the last clicked item in the prefix.
- `y`: integer label (the true next item ID).
- `num_nodes`: number of unique items in the prefix.

The dataset is lazy: graphs are built on demand inside `__getitem__`, so we do not materialize all graphs in memory before training starts.

In [8]:
def build_session_graph(prefix, label):
    """Turn a `(prefix, label)` example into a PyG `Data` graph.

    Nodes are unique items in the prefix. Forward edges follow click order;
    backward edges expose reverse transition context, mirroring the
    SR-GNN / TAGNN incoming/outgoing adjacency channels.
    """
    unique_items = list(dict.fromkeys(prefix))
    item_to_node = {item: index for index, item in enumerate(unique_items)}
    click_sequence = [item_to_node[item] for item in prefix]

    edge_counts = Counter(zip(click_sequence[:-1], click_sequence[1:]))
    out_degree = Counter()
    in_degree = Counter()
    for (source, target), count in edge_counts.items():
        out_degree[source] += count
        in_degree[target] += count

    forward_sources, forward_targets, forward_weights = [], [], []
    backward_sources, backward_targets, backward_weights = [], [], []
    for (source, target), count in edge_counts.items():
        forward_sources.append(source)
        forward_targets.append(target)
        forward_weights.append(count / out_degree[source])

        backward_sources.append(target)
        backward_targets.append(source)
        backward_weights.append(count / in_degree[target])

    if forward_sources:
        forward_edge_index = torch.tensor(
            [forward_sources, forward_targets], dtype=torch.long
        )
        forward_edge_weight = torch.tensor(forward_weights, dtype=torch.float)
        backward_edge_index = torch.tensor(
            [backward_sources, backward_targets], dtype=torch.long
        )
        backward_edge_weight = torch.tensor(backward_weights, dtype=torch.float)
    else:
        forward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        forward_edge_weight = torch.zeros(0, dtype=torch.float)
        backward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        backward_edge_weight = torch.zeros(0, dtype=torch.float)

    return Data(
        x=torch.tensor(unique_items, dtype=torch.long),
        edge_index=forward_edge_index,
        edge_weight=forward_edge_weight,
        forward_edge_index=forward_edge_index,
        forward_edge_weight=forward_edge_weight,
        backward_edge_index=backward_edge_index,
        backward_edge_weight=backward_edge_weight,
        sequence=torch.tensor(click_sequence, dtype=torch.long),
        sequence_length=torch.tensor(len(click_sequence), dtype=torch.long),
        last_click=torch.tensor(click_sequence[-1], dtype=torch.long),
        y=torch.tensor(label, dtype=torch.long),
        num_nodes=len(unique_items),
    )


class SessionGraphDataset(Dataset):
    """Lazy dataset of session-prefix graphs.

    Graphs are built on demand so training does not materialize all PyG
    objects in memory before training starts.
    """

    def __init__(self, prefix_label_rows):
        self.prefix_label_rows = prefix_label_rows

    def __len__(self):
        return len(self.prefix_label_rows)

    def __getitem__(self, index):
        prefix, label = self.prefix_label_rows[index]
        return build_session_graph(prefix, label)

## Vocabulary size

The item embedding table dimension is derived from the maximum remapped item id seen in any train/test row, plus one for padding (id `0`). This must match the convention used by `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`, otherwise the resulting checkpoints and metrics are not directly comparable.

In [9]:
yoochoose_1_64_num_items = vocabulary_size_from_rows(
    YOOCHOOSE_1_64_TRAIN_ROWS, YOOCHOOSE_1_64_TEST_ROWS
)
diginetica_num_items = vocabulary_size_from_rows(
    DIGINETICA_TRAIN_ROWS, DIGINETICA_TEST_ROWS
)

print(f"Yoochoose 1/64 num_items: {yoochoose_1_64_num_items:,}")
print(f"Diginetica      num_items: {diginetica_num_items:,}")

Yoochoose 1/64 num_items: 37,484
Diginetica      num_items: 43,098


## Training and Evaluation Protocol

The training/evaluation contract here intentionally mirrors `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb` so the three branches share the same loop, the same metric definitions, and the same artifact format under `results/` and `results/checkpoints/`.

Setup carried over from SR-GNN / TAGNN (matching the SR-GNN paper protocol):

- Hidden size `100`.
- Random `10%` validation split from the training set.
- Adam, learning rate `0.001`.
- Learning-rate scheduler is selectable via `config["scheduler"]` (Workstream A from `PLAN_SAGPOOL_IMPROVEMENT.md`):
  - `"cosine"` (default, A1): `CosineAnnealingLR(T_max=epochs)`, optional linear warmup of `config["warmup_epochs"]` epochs (A2).
  - `"step5_g05"`: `StepLR(step_size=5, gamma=0.5)` - milder decay than the legacy schedule.
  - `"step3_g01"`: `StepLR(step_size=3, gamma=0.1)` - the legacy SR-GNN/TAGNN-paper schedule, kept for parity runs.
- Default `epochs=30`, `patience=5` (matches the legacy schedule that wins the Workstream A comparison).
- Batch size `100` (sweepable per A5).
- L2 penalty `1e-5` (Workstream C3 sweep: `1e-5 / 5e-5 / 1e-4 / 3e-4`).
- Early stopping on validation `MRR@20`.
- Padding item id `0` is masked out before ranking metrics.

Workstream C regularizers from `PLAN_SAGPOOL_IMPROVEMENT.md` are wired through the same `config` dict, all with safe defaults of `0.0` so a config without C overrides exactly reproduces the pre-Workstream-C numbers:

- **C1** `dropout` (already parametrized): GAT-internal dropout, default `0.1`. Sweep range `0.1 / 0.2 / 0.3 / 0.4`.
- **C2** `embedding_dropout`: per-step Bernoulli mask over rows of the item-embedding table, applied inside `BidirectionalGATEncoder` during training only. Stronger than per-element dropout because every node referencing a dropped item id sees a zero embedding for the whole step.
- **C3** `weight_decay` (already parametrized): Adam L2, default `1e-5`.
- **C4** `label_smoothing`: passed to `F.cross_entropy` in `train_one_epoch` only. Validation/test loss stay unsmoothed so eval numbers are directly comparable across configs.
- **C5** `edge_dropout`: per-step Bernoulli mask over forward and backward session-graph edges (with their corresponding edge weights), train-time only. The encoder is hardened against the degenerate "all edges dropped" case by retaining one edge in that batch.

Helpers exposed (same names as in the other two notebooks for a direct, line-by-line comparable codebase):

- `set_seed`, `get_device`, `random_train_validation_split`, `build_loader`, `mask_padding_item`, `assert_finite`
- `precision_mrr_at_k`, `evaluate`, `train_one_epoch`
- `run_dataset_experiment`

Per-epoch progress is appended to disk after every epoch, so partial runs on Kaggle are recoverable and progress is visible mid-run. Checkpoints are saved each time the validation `MRR@20` improves, plus a final checkpoint at the end of each dataset's training.

In [10]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def random_train_validation_split(rows, validation_fraction=0.1, seed=42):
    indices = list(range(len(rows)))
    random.Random(seed).shuffle(indices)
    validation_size = max(1, int(len(indices) * validation_fraction))
    validation_indices = set(indices[:validation_size])

    train_rows = []
    validation_rows = []
    for index, row in enumerate(rows):
        if index in validation_indices:
            validation_rows.append(row)
        else:
            train_rows.append(row)
    return train_rows, validation_rows


def build_loader(rows, batch_size, shuffle, device):
    dataset = SessionGraphDataset(rows)
    use_cuda = device.type == "cuda"
    num_workers = 2 if use_cuda else 0
    return PyGDataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=use_cuda,
        persistent_workers=num_workers > 0,
    )


def mask_padding_item(logits):
    logits = logits.clone()
    logits[:, 0] = -torch.finfo(logits.dtype).max
    return logits


def assert_finite(name, tensor, batch_index):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"Non-finite {name} detected in batch {batch_index}")

In [11]:
def precision_mrr_at_k(logits, targets, k=20):
    k = min(k, logits.size(1))
    top_items = logits.topk(k, dim=1).indices
    matches = top_items.eq(targets.view(-1, 1))

    hits = matches.any(dim=1).float()
    ranks = torch.zeros(targets.size(0), device=logits.device)
    matched_rows, matched_cols = matches.nonzero(as_tuple=True)
    ranks[matched_rows] = matched_cols.float() + 1
    reciprocal_ranks = torch.where(ranks > 0, 1.0 / ranks, torch.zeros_like(ranks))

    return hits.sum().item(), reciprocal_ranks.sum().item(), targets.size(0)


@torch.no_grad()
def evaluate(model, loader, device, k=20):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    total_hits = 0.0
    total_mrr = 0.0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        logits = mask_padding_item(model(batch))
        loss = F.cross_entropy(logits, batch.y)
        assert_finite("evaluation logits", logits, batch_index)
        assert_finite("evaluation loss", loss, batch_index)

        hits, mrr, examples = precision_mrr_at_k(logits, batch.y, k=k)
        total_loss += loss.item() * examples
        total_examples += examples
        total_hits += hits
        total_mrr += mrr

    return {
        "loss": total_loss / total_examples,
        "precision@20": 100.0 * total_hits / total_examples,
        "mrr@20": 100.0 * total_mrr / total_examples,
        "examples": total_examples,
    }


def train_one_epoch(model, loader, optimizer, device, label_smoothing=0.0):
    """Train one epoch.

    ``label_smoothing`` (Workstream C4 from PLAN_SAGPOOL_IMPROVEMENT.md) is forwarded
    to ``F.cross_entropy``. The default ``0.0`` reproduces the original loss
    exactly. Validation/test losses keep using unsmoothed ``cross_entropy`` so
    the reported eval loss stays comparable across configs.
    """
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        logits = mask_padding_item(model(batch))
        loss = F.cross_entropy(logits, batch.y, label_smoothing=label_smoothing)
        assert_finite("logits", logits, batch_index)
        assert_finite("loss", loss, batch_index)

        loss.backward()
        optimizer.step()

        examples = batch.y.size(0)
        total_loss += loss.item() * examples
        total_examples += examples

    return total_loss / total_examples

In [12]:
DATASETS = {
    "Yoochoose 1/64": {
        "train_rows": YOOCHOOSE_1_64_TRAIN_ROWS,
        "test_rows": YOOCHOOSE_1_64_TEST_ROWS,
        "num_items": yoochoose_1_64_num_items,
    },
    "Diginetica": {
        "train_rows": DIGINETICA_TRAIN_ROWS,
        "test_rows": DIGINETICA_TEST_ROWS,
        "num_items": diginetica_num_items,
    },
}

config = {
    # Run 2 (STATUS-PRAC.md): same arch as Run 1; Diginetica-only reg via PER_DATASET_OVERRIDES.
    # Run 1 used RESULT_FILE_PREFIX=gat_sagpool_v2 (results-sagpool-run1/).
    # sagpool_ratio=0.7, num_layers=2; scheduler step3_g01 (cosine alone underperformed).
    "epochs": 30,
    "patience": 5,
    "batch_size": 100,
    "learning_rate": 0.001,
    "weight_decay": 1e-5,
    "scheduler": "step3_g01",  # one of: "step3_g01" (default), "step5_g05", "cosine"
    "warmup_epochs": 0,         # linear warmup before the main schedule (A2)
    "lr_decay_step": 3,         # used only when scheduler == "step3_g01"
    "lr_decay_gamma": 0.1,      # used only when scheduler == "step3_g01"
    "validation_fraction": 0.1,
    "hidden_dim": 100,         # Workstream B6 sweep: 100 / 128 / 200
    "num_layers": 2,           # Run 1/2 arch (phase2b)
    "num_heads": 4,            # Workstream B2 sweep: 4 / 8 (needs hidden_dim % heads == 0)
    "dropout": 0.1,            # Workstream C1 sweep: 0.1 / 0.2 / 0.3 / 0.4
    "sagpool_ratio": 0.7,      # Run 1/2 arch (phase2b b4_ratio07)
    "sagpool_blocks": 1,       # Workstream B3 sweep: 1 (current) / 2
    "sagpool_scorer": "graph_conv",  # Workstream B5 sweep: "graph_conv" / "gat"
    "readout_combine": "linear",     # Workstream B7 sweep: "linear" / "gated"
    "label_smoothing": 0.0,    # Workstream C4 sweep: 0.0 / 0.05 / 0.1 / 0.2
    "embedding_dropout": 0.0,  # Workstream C2 sweep: 0.0 / 0.05 / 0.1 (drop full item rows)
    "edge_dropout": 0.0,       # Workstream C5 sweep: 0.0 / 0.1 / 0.2 / 0.3 (train-time only)
    "seed": 42,
}

MODEL_NAME = "GATSAGPool"
RESULT_FILE_PREFIX = "gat_sagpool_v2_digi_reg"  # Run 2 — Diginetica reg overrides only

# Run 2: stronger regularization on Diginetica only (Phase 3C hints; no label_smoothing).
PER_DATASET_OVERRIDES = {
    "Diginetica": {
        "dropout": 0.2,
        "embedding_dropout": 0.1,
        "weight_decay": 1e-4,
    },
}


def config_for_dataset(dataset_name, base_config=None):
    """Merge global config with optional per-dataset overrides (Run 2)."""
    merged = dict(base_config if base_config is not None else config)
    merged.update(PER_DATASET_OVERRIDES.get(dataset_name, {}))
    return merged


device = get_device()
print(f"device={device}")
print("global config:", config)
print("PER_DATASET_OVERRIDES:", PER_DATASET_OVERRIDES)
for _ds in DATASETS:
    print(f"  {_ds}:", config_for_dataset(_ds))

device=cuda
global config: {'epochs': 30, 'patience': 5, 'batch_size': 100, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'scheduler': 'step3_g01', 'warmup_epochs': 0, 'lr_decay_step': 3, 'lr_decay_gamma': 0.1, 'validation_fraction': 0.1, 'hidden_dim': 100, 'num_layers': 2, 'num_heads': 4, 'dropout': 0.1, 'sagpool_ratio': 0.7, 'sagpool_blocks': 1, 'sagpool_scorer': 'graph_conv', 'readout_combine': 'linear', 'label_smoothing': 0.0, 'embedding_dropout': 0.0, 'edge_dropout': 0.0, 'seed': 42}
PER_DATASET_OVERRIDES: {'Diginetica': {'dropout': 0.2, 'embedding_dropout': 0.1, 'weight_decay': 0.0001}}
  Yoochoose 1/64: {'epochs': 30, 'patience': 5, 'batch_size': 100, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'scheduler': 'step3_g01', 'warmup_epochs': 0, 'lr_decay_step': 3, 'lr_decay_gamma': 0.1, 'validation_fraction': 0.1, 'hidden_dim': 100, 'num_layers': 2, 'num_heads': 4, 'dropout': 0.1, 'sagpool_ratio': 0.7, 'sagpool_blocks': 1, 'sagpool_scorer': 'graph_conv', 'readout_combine': 'linea

In [13]:
def checkpoint_name(dataset_name, prefix=None):
    """Build a per-dataset checkpoint filename.

    The optional ``prefix`` keyword argument lets sweep runs keep their
    checkpoints from colliding with each other (each variant uses its own
    ``result_file_prefix``). When ``prefix`` is ``None`` the global
    ``RESULT_FILE_PREFIX`` is used, preserving the original behavior.
    """
    if prefix is None:
        prefix = RESULT_FILE_PREFIX
    safe_name = dataset_name.lower().replace(" ", "_").replace("/", "_")
    return f"{prefix}_{safe_name}.pt"


def _save_history_so_far(history_rows, history_path):
    pd.DataFrame(history_rows).to_csv(history_path, index=False)


def build_scheduler(optimizer, config):
    """Build the LR scheduler per Workstream A from PLAN_SAGPOOL_IMPROVEMENT.md.

    Choices via ``config["scheduler"]``:
      - ``"cosine"``    : ``CosineAnnealingLR(T_max=epochs - warmup_epochs)``.
                          When ``config["warmup_epochs"] > 0`` a linear warmup is
                          prepended via ``SequentialLR(LinearLR + CosineAnnealingLR)``.
      - ``"step5_g05"`` : ``StepLR(step_size=5, gamma=0.5)`` - milder than legacy.
      - ``"step3_g01"`` : ``StepLR(step_size=config["lr_decay_step"],
                                   gamma=config["lr_decay_gamma"])``.
                          Kept for parity with the legacy SR-GNN/TAGNN-paper runs.
    """
    name = config.get("scheduler", "cosine")
    warmup = int(config.get("warmup_epochs", 0))
    epochs = int(config["epochs"])

    if name == "step3_g01":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=config["lr_decay_step"],
            gamma=config["lr_decay_gamma"],
        )
    if name == "step5_g05":
        return torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=5, gamma=0.5,
        )
    if name == "cosine":
        cosine_T = max(1, epochs - warmup)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cosine_T,
        )
        if warmup <= 0:
            return cosine
        warmup_sched = torch.optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor=1e-3,
            end_factor=1.0,
            total_iters=warmup,
        )
        return torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[warmup_sched, cosine],
            milestones=[warmup],
        )
    raise ValueError(f"unknown scheduler: {name!r}")


def run_dataset_experiment(
    dataset_name,
    dataset,
    config,
    device,
    model_factory,
    model_name=MODEL_NAME,
    result_file_prefix=RESULT_FILE_PREFIX,
):
    """Train a session-graph model on a single dataset.

    `model_factory(num_items, config)` must return an `nn.Module` that
    accepts a PyG batch (with the fields produced by `build_session_graph`)
    and returns logits of shape `[batch_size, num_items]`. The runner is
    intentionally model-agnostic so SR-GNN, TAGNN, and SAGPool variants
    share the same training harness.
    """
    set_seed(config["seed"])
    train_rows = list(dataset["train_rows"])
    test_rows = list(dataset["test_rows"])
    train_rows, validation_rows = random_train_validation_split(
        train_rows,
        validation_fraction=config["validation_fraction"],
        seed=config["seed"],
    )

    train_loader = build_loader(
        train_rows, config["batch_size"], shuffle=True, device=device
    )
    validation_loader = build_loader(
        validation_rows, config["batch_size"], shuffle=False, device=device
    )
    test_loader = build_loader(
        test_rows, config["batch_size"], shuffle=False, device=device
    )

    model = model_factory(num_items=dataset["num_items"], config=config).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = build_scheduler(optimizer, config)

    history = []
    best_validation_mrr = -1.0
    best_validation_precision = -1.0
    best_epoch = 0
    best_state = None
    bad_counter = 0

    checkpoint_path = CHECKPOINTS_DIR / checkpoint_name(dataset_name, prefix=result_file_prefix)
    history_path = RESULTS_DIR / f"{result_file_prefix}_history.csv"

    for epoch in range(1, config["epochs"] + 1):
        started_at = time.time()
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device,
            label_smoothing=config.get("label_smoothing", 0.0),
        )
        validation_metrics = evaluate(model, validation_loader, device)
        scheduler.step()

        row = {
            "dataset": dataset_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_metrics["loss"],
            "validation_precision@20": validation_metrics["precision@20"],
            "validation_mrr@20": validation_metrics["mrr@20"],
            "epoch_seconds": time.time() - started_at,
            "train_examples": len(train_rows),
            "validation_examples": len(validation_rows),
            "test_examples": len(test_rows),
        }
        history.append(row)
        print(
            f"{dataset_name} epoch {epoch:02d} "
            f"loss={train_loss:.4f} "
            f"val_loss={row['validation_loss']:.4f} "
            f"val_P@20={row['validation_precision@20']:.2f} "
            f"val_MRR@20={row['validation_mrr@20']:.2f} "
            f"time={row['epoch_seconds']:.1f}s"
        )

        _save_history_so_far(history, history_path)

        if validation_metrics["mrr@20"] >= best_validation_mrr:
            best_validation_mrr = validation_metrics["mrr@20"]
            best_validation_precision = validation_metrics["precision@20"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            bad_counter = 0
            torch.save(
                {
                    "dataset": dataset_name,
                    "model": model_name,
                    "config": dict(config),
                    "num_items": dataset["num_items"],
                    "state_dict": best_state,
                    "best_epoch": best_epoch,
                    "best_validation_precision@20": best_validation_precision,
                    "best_validation_mrr@20": best_validation_mrr,
                },
                checkpoint_path,
            )
        else:
            bad_counter += 1
            if bad_counter >= config["patience"]:
                print(f"early stopping at epoch {epoch}; best epoch was {best_epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, device)

    torch.save(
        {
            "dataset": dataset_name,
            "model": model_name,
            "config": dict(config),
            "num_items": dataset["num_items"],
            "state_dict": model.state_dict(),
            "test_metrics": test_metrics,
            "history": history,
            "best_epoch": best_epoch,
            "best_validation_precision@20": best_validation_precision,
            "best_validation_mrr@20": best_validation_mrr,
        },
        checkpoint_path,
    )

    result = {
        "dataset": dataset_name,
        "test_precision@20": test_metrics["precision@20"],
        "test_mrr@20": test_metrics["mrr@20"],
        "test_loss": test_metrics["loss"],
        "best_epoch": best_epoch,
        "best_validation_precision@20": best_validation_precision,
        "best_validation_mrr@20": best_validation_mrr,
        "train_examples": len(train_rows),
        "validation_examples": len(validation_rows),
        "test_examples": len(test_rows),
        "num_items": dataset["num_items"],
        "checkpoint_path": str(checkpoint_path),
    }
    return result, history

## Bidirectional GAT Encoder

The encoder is a direct port of the bidirectional GAT stack used by `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`. Keeping the encoder identical across the three branches isolates the architectural difference of this notebook to the readout (SAGPool), so any change in `Precision@20` / `MRR@20` is attributable to the readout choice rather than to encoder differences.

Defaults match the other two notebooks:

- hidden size `100`
- one GAT layer per direction
- `4` attention heads, per-head output `25`, concatenated back to `100`
- dropout `0.1` inside the GAT layer and after activation
- ELU activation
- residual connection inside each directional stack
- self-loops added by `GATConv`
- normalized transition counts passed as one-dimensional edge attributes
- forward / backward states concatenated, then fused by a linear `200 -> 100`

In [14]:
class DirectionalGATStack(nn.Module):
    """GAT stack for one transition direction."""

    def __init__(
        self,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
        concat_heads=True,
        residual=True,
    ):
        super().__init__()

        if concat_heads and hidden_dim % num_heads != 0:
            raise ValueError(
                f"hidden_dim ({hidden_dim}) must be divisible by num_heads "
                f"({num_heads}) when concat_heads=True"
            )

        self.dropout = dropout
        self.residual = residual
        per_head_dim = hidden_dim // num_heads if concat_heads else hidden_dim
        self.gat_layers = nn.ModuleList(
            GATConv(
                in_channels=hidden_dim,
                out_channels=per_head_dim,
                heads=num_heads,
                concat=concat_heads,
                dropout=dropout,
                add_self_loops=True,
                edge_dim=1,
            )
            for _ in range(num_layers)
        )

    def forward(self, node_features, edge_index, edge_weight=None):
        edge_attr = edge_weight.unsqueeze(-1) if edge_weight is not None else None
        for gat_layer in self.gat_layers:
            previous = node_features
            node_features = F.elu(gat_layer(node_features, edge_index, edge_attr))
            node_features = F.dropout(
                node_features, p=self.dropout, training=self.training
            )
            if self.residual:
                node_features = node_features + previous
        return node_features


class BidirectionalGATEncoder(nn.Module):
    """Separate forward/backward GAT stacks with shared item embeddings.

    Workstream C (PLAN_SAGPOOL_IMPROVEMENT.md) wires two graph-specific
    regularizers into this encoder, both train-time-only:

      - C2 ``embedding_dropout`` : per-step Bernoulli mask over rows of the
        item-embedding matrix (Merity et al. 2018 "embedded dropout"). When
        a row is dropped, every node referencing that item id sees a zero
        embedding for the whole step, which is stronger than dropping random
        elements of individual lookups.
      - C5 ``edge_dropout`` : per-step Bernoulli mask over forward and
        backward session-graph edges (with their corresponding edge weights).
        Forces the GAT layers to attend over varying sub-graphs and reduces
        co-adaptation to specific transitions.

    Defaults are ``0.0`` for both, so a config without these keys reproduces
    the original (pre-Workstream C) encoder behavior exactly.
    """

    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
        embedding_dropout=0.0,
        edge_dropout=0.0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.forward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, dropout
        )
        self.backward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, dropout
        )
        self.direction_fusion = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)
        self.embedding_dropout = float(embedding_dropout)
        self.edge_dropout = float(edge_dropout)

    def _embed(self, node_item_ids):
        weight = self.embedding.weight
        if self.training and self.embedding_dropout > 0.0:
            keep = 1.0 - self.embedding_dropout
            row_mask = weight.new_empty((weight.size(0), 1)).bernoulli_(keep)
            row_mask = row_mask / max(keep, 1e-12)
            weight = weight * row_mask
        return F.embedding(node_item_ids, weight, padding_idx=self.embedding.padding_idx)

    def _maybe_drop_edges(self, edge_index, edge_weight):
        if (
            not self.training
            or self.edge_dropout <= 0.0
            or edge_index.size(1) == 0
        ):
            return edge_index, edge_weight
        keep = 1.0 - self.edge_dropout
        keep_mask = torch.rand(edge_index.size(1), device=edge_index.device) < keep
        if not bool(keep_mask.any()):
            # Degenerate case: every edge dropped. Keep one edge to avoid an
            # empty edge_index that would yield zero attention contributions.
            keep_mask[0] = True
        new_edge_index = edge_index[:, keep_mask]
        new_edge_weight = (
            edge_weight[keep_mask] if edge_weight is not None else None
        )
        return new_edge_index, new_edge_weight

    def forward(
        self,
        node_item_ids,
        forward_edge_index,
        forward_edge_weight,
        backward_edge_index,
        backward_edge_weight,
    ):
        node_features = self._embed(node_item_ids)
        forward_edge_index, forward_edge_weight = self._maybe_drop_edges(
            forward_edge_index, forward_edge_weight
        )
        backward_edge_index, backward_edge_weight = self._maybe_drop_edges(
            backward_edge_index, backward_edge_weight
        )
        forward_features = self.forward_gat(
            node_features, forward_edge_index, forward_edge_weight
        )
        backward_features = self.backward_gat(
            node_features, backward_edge_index, backward_edge_weight
        )
        return self.direction_fusion(
            torch.cat([forward_features, backward_features], dim=-1)
        )

## GAT-SAGPool Model

Encoder is `BidirectionalGATEncoder` from above; readout is a SAGPool head (one or more hierarchical blocks) with last-click signal preservation. All Workstream B (PLAN_SAGPOOL_IMPROVEMENT.md) knobs are wired through the shared `config` dict, with defaults that exactly reproduce the original single-block / `GraphConv`-scorer / linear-projection variant.

### Readout

1. Score every node with a scoring GNN (`config["sagpool_scorer"]`):
   - `"graph_conv"` (default): the standard SAGPool scorer.
   - `"gat"` (B5): single-head `GATConv` with `add_self_loops=False`, attention-based scoring.
2. Retain the top fraction of nodes per graph (per-graph top-k), using `config["sagpool_ratio"]` (B4, default `0.5`). PyG's `SAGPooling` guarantees that every graph keeps at least one node, including degenerate cases like single-item prefixes.
3. Aggregate the retained nodes per graph by concatenating `global_mean_pool` and `global_max_pool` outputs (`2 * hidden_dim`).
4. (B3 — hierarchical SAGPool) When `config["sagpool_blocks"] > 1`, repeat steps 1–3 over the pooled graph. A `GraphConv` layer refreshes node features between successive pools so each block scores nodes against an updated representation. The per-block mean/max readouts are then summed (JK-net style) into a single `[B, 2H]` pooled signal.
5. Concatenate the aggregated SAGPool readout with the **last-click contextual node embedding** taken from the un-pooled encoder output. SAGPool is permutation invariant by construction and does not see ordering information, so explicitly carrying the last-click vector is what protects the immediate intent signal of the session. The last-click vector is sourced from the pre-pool features so it survives even aggressive top-k pruning.
6. Combine via `config["readout_combine"]`:
   - `"linear"` (default): `Linear(3H -> H)` over `[mean || max || last_click]`.
   - `"gated"` (B7): a learned per-feature gate blends the projected pooled signal with the projected last-click signal, allowing the model to down-weight noisy pooled representations on short sessions.

### Scoring head

The session embedding is multiplied by the shared item embedding matrix to produce logits of shape `[batch_size, num_items]`. The padding item id `0` is excluded from ranking by `mask_padding_item` (defined earlier and shared across the three notebooks), so the candidate space is exactly the remapped train item ids `1 .. num_items - 1`. The padding row of the embedding table is also explicitly zeroed in `reset_parameters` and held at zero by `padding_idx=0` on `nn.Embedding`.

In [15]:
class GATSAGPool(nn.Module):
    """Bidirectional GAT encoder + SAGPool readout for session recommendation.

    Workstream B (PLAN_SAGPOOL_IMPROVEMENT.md) parametrizes:
      - B1 ``num_layers``         : depth of the GAT encoder per direction.
      - B2 ``num_heads``          : multi-head count in the GAT encoder.
      - B3 ``sagpool_blocks``     : number of hierarchical SAGPool blocks. With
                                    ``blocks > 1`` an intermediate ``GraphConv``
                                    refines node features between successive pools
                                    (SAGPool-paper hierarchy). The mean+max readouts
                                    of all blocks are summed (JK-net style).
      - B4 ``sagpool_ratio``      : top-k retention ratio per block.
      - B5 ``sagpool_scorer``     : ``"graph_conv"`` (default) or ``"gat"`` (single
                                    head, no self-loops, attention-based scoring).
      - B6 ``hidden_dim``         : embedding / hidden width.
      - B7 ``readout_combine``    : ``"linear"`` (default, ``Linear(3H -> H)`` over
                                    ``[mean || max || last_click]``) or ``"gated"``
                                    (learned per-feature gate that blends the pooled
                                    signal with the last-click signal).

    Default values reproduce the original single-block / GraphConv-scorer / linear
    readout architecture exactly, so a config without any B overrides yields the
    same model as before.

    Architecture summary (single block, defaults):

      session prefix
          |
      directed session graph (forward + backward edges, normalized weights)
          |
      BidirectionalGATEncoder  ->  contextual node embeddings  [N, H]
          |                                          \\
          |                                           +--> last_click_repr [B, H]
          v
      SAGPooling (scorer = graph_conv | gat, top-`ratio`)  ->  pooled nodes
          |  (repeat ``sagpool_blocks`` times, with GraphConv refinement in between)
      sum_b [global_mean_pool || global_max_pool]_b  ->  pooled session repr [B, 2H]
          |
      readout_combine([pooled_session, last_click_repr])  ->  session_repr [B, H]
          |
      session_repr @ item_embedding.T  ->  logits [B, num_items]
    """

    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
        sagpool_ratio=0.5,
        sagpool_blocks=1,
        sagpool_scorer="graph_conv",
        readout_combine="linear",
        embedding_dropout=0.0,
        edge_dropout=0.0,
    ):
        super().__init__()
        self.encoder = BidirectionalGATEncoder(
            num_items,
            hidden_dim,
            num_layers,
            num_heads,
            dropout,
            embedding_dropout=embedding_dropout,
            edge_dropout=edge_dropout,
        )
        self.hidden_dim = hidden_dim
        self.sagpool_ratio = sagpool_ratio
        self.sagpool_blocks = int(sagpool_blocks)
        if self.sagpool_blocks < 1:
            raise ValueError(
                f"sagpool_blocks must be >= 1, got {self.sagpool_blocks}"
            )
        if readout_combine not in ("linear", "gated"):
            raise ValueError(
                f"readout_combine must be 'linear' or 'gated', got {readout_combine!r}"
            )
        self.readout_combine = readout_combine

        if sagpool_scorer == "graph_conv":
            scorer_gnn = GraphConv
            scorer_kwargs = {}
        elif sagpool_scorer == "gat":
            scorer_gnn = GATConv
            # Single-head, no self-loops: keeps the score map compatible with
            # SAGPooling's expectation of a ``[N, 1]`` score tensor and avoids
            # numerical issues with isolated nodes in tiny session graphs.
            scorer_kwargs = {"heads": 1, "add_self_loops": False}
        else:
            raise ValueError(
                f"sagpool_scorer must be 'graph_conv' or 'gat', got {sagpool_scorer!r}"
            )
        self.sagpool_scorer = sagpool_scorer

        self.pool_blocks = nn.ModuleList(
            [
                SAGPooling(
                    in_channels=hidden_dim,
                    ratio=sagpool_ratio,
                    GNN=scorer_gnn,
                    **scorer_kwargs,
                )
                for _ in range(self.sagpool_blocks)
            ]
        )
        # Intermediate GraphConv between successive pools (only when blocks > 1).
        self.between_block_convs = nn.ModuleList(
            [GraphConv(hidden_dim, hidden_dim) for _ in range(self.sagpool_blocks - 1)]
        )

        if readout_combine == "linear":
            self.session_projection = nn.Linear(3 * hidden_dim, hidden_dim, bias=True)
        else:  # gated
            self.gate_pooled = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)
            self.gate_last = nn.Linear(hidden_dim, hidden_dim, bias=True)
            self.gate_compute = nn.Linear(3 * hidden_dim, hidden_dim, bias=True)

        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_dim)
        for parameter in self.parameters():
            parameter.data.uniform_(-stdv, stdv)
        with torch.no_grad():
            self.encoder.embedding.weight[0].fill_(0)

    def _readout(self, mean_session, max_session, last_click_repr):
        if self.readout_combine == "linear":
            return self.session_projection(
                torch.cat([mean_session, max_session, last_click_repr], dim=-1)
            )
        # Gated mixing: a learned per-feature gate blends the pooled-session signal
        # (a projection of [mean || max]) with the last-click signal.
        pooled_signal = self.gate_pooled(torch.cat([mean_session, max_session], dim=-1))
        last_signal = self.gate_last(last_click_repr)
        gate = torch.sigmoid(
            self.gate_compute(
                torch.cat([mean_session, max_session, last_click_repr], dim=-1)
            )
        )
        return gate * pooled_signal + (1.0 - gate) * last_signal

    def forward(self, batch):
        """Return logits `[batch_size, num_items]`."""
        node_hidden = self.encoder(
            batch.x,
            batch.forward_edge_index,
            batch.forward_edge_weight,
            batch.backward_edge_index,
            batch.backward_edge_weight,
        )

        # Last-click representation is taken from the pre-pool node features so the
        # signal survives even aggressive top-k pruning (it would otherwise be lost
        # if the last-click node was dropped by SAGPool).
        graph_node_offsets = batch.ptr[:-1]
        last_click_local = batch.last_click.view(-1).long()
        last_click_abs = graph_node_offsets + last_click_local
        last_click_repr = node_hidden[last_click_abs]

        x = node_hidden
        edge_index = batch.forward_edge_index
        node_batch = batch.batch
        block_means = []
        block_maxes = []
        for block_idx, pool in enumerate(self.pool_blocks):
            if block_idx > 0:
                # Refresh node features between successive pools so each subsequent
                # SAGPool block scores nodes against an updated representation.
                x = self.between_block_convs[block_idx - 1](x, edge_index)
                x = F.relu(x)
            pooled_x, pooled_ei, _, pooled_batch, _, _ = pool(
                x, edge_index, None, node_batch
            )
            block_means.append(
                global_mean_pool(pooled_x, pooled_batch, size=batch.num_graphs)
            )
            block_maxes.append(
                global_max_pool(pooled_x, pooled_batch, size=batch.num_graphs)
            )
            x = pooled_x
            edge_index = pooled_ei
            node_batch = pooled_batch

        # JK-net style aggregation of per-block readouts.
        mean_session = torch.stack(block_means, dim=0).sum(dim=0)
        max_session = torch.stack(block_maxes, dim=0).sum(dim=0)

        session_repr = self._readout(mean_session, max_session, last_click_repr)

        item_embeddings = self.encoder.embedding.weight
        logits = session_repr @ item_embeddings.T
        return logits

In [16]:
def gat_sagpool_factory(num_items, config):
    """Build a `GATSAGPool` model from the shared experiment config dict.

    Used by `run_dataset_experiment(..., model_factory=gat_sagpool_factory)`.
    Workstream B knobs (PLAN_SAGPOOL_IMPROVEMENT.md) are read with sensible
    defaults, so older configs keep working unchanged.
    """
    return GATSAGPool(
        num_items=num_items,
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        dropout=config["dropout"],
        sagpool_ratio=config.get("sagpool_ratio", 0.5),
        sagpool_blocks=config.get("sagpool_blocks", 1),
        sagpool_scorer=config.get("sagpool_scorer", "graph_conv"),
        readout_combine=config.get("readout_combine", "linear"),
        embedding_dropout=config.get("embedding_dropout", 0.0),
        edge_dropout=config.get("edge_dropout", 0.0),
    )


config.setdefault("sagpool_ratio", 0.7)
config.setdefault("sagpool_blocks", 1)
config.setdefault("sagpool_scorer", "graph_conv")
config.setdefault("readout_combine", "linear")
config.setdefault("embedding_dropout", 0.0)
config.setdefault("edge_dropout", 0.0)
config.setdefault("label_smoothing", 0.0)

0.0

## Numerical stability and shape validation

Before launching a full training run on Kaggle (which is expensive), this section runs an inexpensive smoke test on a handcrafted mini-batch covering the edge cases of the session-graph dataset:

- a normal length-5 prefix
- a prefix with a repeated item
- a length-1 prefix (minimum graph: 1 node, 0 edges)
- a prefix containing a cycle / repeated transitions
- a longer length-8 prefix

For this batch the smoke test asserts:

- the per-graph contract from `build_session_graph` (1D `x`, two-row `forward_edge_index` / `backward_edge_index`, scalar `last_click` matching `sequence[-1]`)
- the per-batch contract from PyG batching (`batch.num_graphs`, `batch.y`, `batch.last_click`, `batch.sequence_length` are all `[batch_size]`)
- the model forward pass returns logits of shape `[batch_size, num_items]`
- all logits are finite
- after `mask_padding_item`, the padding id `0` is never selected by `argmax`
- the backward pass works for variable-size graphs and produces a finite loss

If any assertion trips, training is aborted immediately rather than burning Kaggle GPU time on a broken model.

In [17]:
def _run_gat_sagpool_smoke_test():
    set_seed(0)

    smoke_num_items = 50
    sample_rows = [
        ([1, 2, 3, 4, 5], 6),
        ([1, 1, 2], 3),
        ([7], 8),
        ([1, 2, 3, 1, 2], 4),
        ([5, 6, 7, 8, 9, 10, 11, 12], 13),
    ]
    batch_size = len(sample_rows)

    single = build_session_graph(*sample_rows[0])
    assert single.x.dim() == 1, f"x must be 1D, got {tuple(single.x.shape)}"
    assert single.forward_edge_index.size(0) == 2, "forward_edge_index must have 2 rows"
    assert single.backward_edge_index.size(0) == 2, "backward_edge_index must have 2 rows"
    assert single.num_nodes == single.x.numel(), "num_nodes must equal len(x)"
    assert int(single.last_click.item()) == int(single.sequence[-1].item()), (
        "last_click must match the last position of the sequence"
    )

    dataset = SessionGraphDataset(sample_rows)
    loader = PyGDataLoader(dataset, batch_size=batch_size, shuffle=False)
    batch = next(iter(loader)).to(device)

    assert batch.num_graphs == batch_size, batch.num_graphs
    assert tuple(batch.y.shape) == (batch_size,), batch.y.shape
    assert tuple(batch.last_click.shape) == (batch_size,), batch.last_click.shape
    assert tuple(batch.sequence_length.shape) == (batch_size,), batch.sequence_length.shape

    model = gat_sagpool_factory(num_items=smoke_num_items, config=config).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(batch)
        masked = mask_padding_item(logits)

    assert tuple(logits.shape) == (batch_size, smoke_num_items), (
        f"expected {(batch_size, smoke_num_items)}, got {tuple(logits.shape)}"
    )
    assert torch.isfinite(logits).all(), "non-finite logits in smoke test"
    assert torch.isfinite(masked).all(), "non-finite masked logits in smoke test"

    top_per_row = masked.argmax(dim=1)
    assert (top_per_row > 0).all(), "argmax must never pick padding id 0 after masking"

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    optimizer.zero_grad(set_to_none=True)
    train_logits = mask_padding_item(model(batch))
    loss = F.cross_entropy(train_logits, batch.y)
    assert torch.isfinite(loss), f"non-finite training loss: {loss}"
    loss.backward()
    optimizer.step()

    print(
        f"smoke_test_passed batch={batch_size} num_items={smoke_num_items} "
        f"logits_shape={tuple(logits.shape)} backward_loss={loss.item():.4f}"
    )


_run_gat_sagpool_smoke_test()

/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


smoke_test_passed batch=5 num_items=50 logits_shape=(5, 50) backward_loss=3.8970


## Training Run 

This section actually trains `GATSAGPool` on `Yoochoose 1/64` and `Diginetica` using the model-agnostic `run_dataset_experiment`.

- **C1 - training schedule**: Adam with `lr=0.001`, `weight_decay=1e-5`, batch size `100`, default `epochs=30`, `patience=5`, `scheduler="step3_g01"` (`StepLR(step_size=3, gamma=0.1)`). The Workstream A cosine variant is still wired through `config["scheduler"]` (`"cosine"` / `"step5_g05"`, with optional `config["warmup_epochs"]`) but underperformed the legacy schedule on its own and should only be re-tried alongside the Workstream C regularizers. Workstream B sweeps inherit this legacy schedule by default so each architecture variant is evaluated on the same known-good baseline.
- **C2 - validation protocol**: deterministic `10%` random split from the training rows via `random_train_validation_split`. Each epoch records `train_loss`, `validation_loss`, `validation_precision@20`, and `validation_mrr@20` into the per-dataset history CSV.
- **C3 - checkpointing**: best epoch is selected by `validation_mrr@20` (consistent with the SR-GNN / TAGNN notebooks). Checkpoints use `RESULT_FILE_PREFIX` (Run 2: `gat_sagpool_v2_digi_reg_<dataset>.pt`).

- **Run 2**: Yoochoose uses the global `config`; Diginetica merges `PER_DATASET_OVERRIDES` (dropout / embedding dropout / weight decay) via `config_for_dataset` before `run_dataset_experiment`.

Per-epoch history is streamed to `results/{RESULT_FILE_PREFIX}_history.csv` after every epoch, so a partial Kaggle run is still useful.

In [18]:
all_results = []
all_history = []

for dataset_name, dataset in DATASETS.items():
    dataset_config = config_for_dataset(dataset_name)
    print(f"=== training {dataset_name} ===")
    if dataset_name in PER_DATASET_OVERRIDES:
        print(f"    overrides: {PER_DATASET_OVERRIDES[dataset_name]}")
    result, history = run_dataset_experiment(
        dataset_name, dataset, dataset_config, device, gat_sagpool_factory
    )
    all_results.append(result)
    all_history.extend(history)
    print(
        f"=== finished {dataset_name}: "
        f"test_P@20={result['test_precision@20']:.2f} "
        f"test_MRR@20={result['test_mrr@20']:.2f} "
        f"best_epoch={result['best_epoch']} "
        f"checkpoint={result['checkpoint_path']} ==="
    )

results = pd.DataFrame(all_results)
history = pd.DataFrame(all_history)

results_path = RESULTS_DIR / f"{RESULT_FILE_PREFIX}_results.csv"
history_path = RESULTS_DIR / f"{RESULT_FILE_PREFIX}_history.csv"
results.to_csv(results_path, index=False)
history.to_csv(history_path, index=False)

try:
    display(results)
except NameError:
    print(results.to_string(index=False))
print(f"saved {results_path}")
print(f"saved {history_path}")

=== training Yoochoose 1/64 ===
Yoochoose 1/64 epoch 01 loss=5.4943 val_loss=4.7291 val_P@20=64.75 val_MRR@20=28.65 time=140.7s
Yoochoose 1/64 epoch 02 loss=4.4747 val_loss=4.5125 val_P@20=67.68 val_MRR@20=30.12 time=139.6s
Yoochoose 1/64 epoch 03 loss=4.2192 val_loss=4.4555 val_P@20=68.08 val_MRR@20=30.63 time=140.1s
Yoochoose 1/64 epoch 04 loss=3.7745 val_loss=4.3547 val_P@20=69.48 val_MRR@20=32.45 time=141.1s
Yoochoose 1/64 epoch 05 loss=3.6781 val_loss=4.3661 val_P@20=69.34 val_MRR@20=32.58 time=145.7s
Yoochoose 1/64 epoch 06 loss=3.6334 val_loss=4.3847 val_P@20=69.44 val_MRR@20=32.59 time=145.4s
Yoochoose 1/64 epoch 07 loss=3.5461 val_loss=4.3933 val_P@20=69.34 val_MRR@20=32.77 time=141.1s
Yoochoose 1/64 epoch 08 loss=3.5369 val_loss=4.3994 val_P@20=69.37 val_MRR@20=32.79 time=137.8s
Yoochoose 1/64 epoch 09 loss=3.5320 val_loss=4.4031 val_P@20=69.37 val_MRR@20=32.79 time=138.1s
Yoochoose 1/64 epoch 10 loss=3.5205 val_loss=4.4040 val_P@20=69.35 val_MRR@20=32.78 time=139.7s
Yoochoos

,dataset,test_precision@20,test_mrr@20,test_loss,best_epoch,best_validation_precision@20,best_validation_mrr@20,train_examples,validation_examples,test_examples,num_items,checkpoint_path
0,Yoochoose 1/64,70.272639,31.079248,4.319441,8,69.368663,32.794825,332874,36985,55898,37484,/kaggle/working/results/checkpoints/gat_sagpoo...
1,Diginetica,48.769266,15.855698,5.642330,11,54.003642,18.602830,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sagpoo...


saved /kaggle/working/results/gat_sagpool_v2_digi_reg_results.csv
saved /kaggle/working/results/gat_sagpool_v2_digi_reg_history.csv


## Checkpoint Summary

For each dataset this writes a compact summary that combines:

- `best_epoch` (the epoch that won the validation `MRR@20` competition),
- the best validation `Precision@20` and `MRR@20` at that epoch,
- the final test `Precision@20`, `MRR@20`, and `loss` (evaluated after restoring the best weights),
- the path to the best checkpoint on disk.

The same physical file holds both the best-by-validation weights and the final test-time metrics (the runner restores best weights before computing test metrics, then overwrites the checkpoint with the best state and those test metrics attached). Saving an explicit summary CSV makes the "where is the best model" registry explicit and easy to consume from the cross-notebook comparison tables under `results/`.

In [19]:
checkpoint_summary_rows = []
for record in all_results:
    checkpoint_summary_rows.append(
        {
            "dataset": record["dataset"],
            "best_epoch": record["best_epoch"],
            "best_validation_precision@20": record["best_validation_precision@20"],
            "best_validation_mrr@20": record["best_validation_mrr@20"],
            "test_precision@20": record["test_precision@20"],
            "test_mrr@20": record["test_mrr@20"],
            "test_loss": record["test_loss"],
            "best_checkpoint_path": record["checkpoint_path"],
            "final_checkpoint_path": record["checkpoint_path"],
        }
    )

checkpoint_summary = pd.DataFrame(checkpoint_summary_rows)
checkpoint_summary_path = (
    RESULTS_DIR / f"{RESULT_FILE_PREFIX}_checkpoint_summary.csv"
)
checkpoint_summary.to_csv(checkpoint_summary_path, index=False)

try:
    display(checkpoint_summary)
except NameError:
    print(checkpoint_summary.to_string(index=False))
print(f"saved {checkpoint_summary_path}")

,dataset,best_epoch,best_validation_precision@20,best_validation_mrr@20,test_precision@20,test_mrr@20,test_loss,best_checkpoint_path,final_checkpoint_path
0,Yoochoose 1/64,8,69.368663,32.794825,70.272639,31.079248,4.319441,/kaggle/working/results/checkpoints/gat_sagpoo...,/kaggle/working/results/checkpoints/gat_sagpoo...
1,Diginetica,11,54.003642,18.602830,48.769266,15.855698,5.642330,/kaggle/working/results/checkpoints/gat_sagpoo...,/kaggle/working/results/checkpoints/gat_sagpoo...


saved /kaggle/working/results/gat_sagpool_v2_digi_reg_checkpoint_summary.csv


## Reporting


This is the reporting layer for `GATSAGPool`.



The cross-model writes are intentionally non-invasive: they never delete other models' rows and never touch a file unless its schema is recognized.

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results.plot.bar(
    x="dataset",
    y="test_precision@20",
    ax=axes[0],
    legend=False,
    color="#3b6ea8",
    title="Test Precision@20 (GAT-SAGPool)",
)
axes[0].set_ylabel("%")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

results.plot.bar(
    x="dataset",
    y="test_mrr@20",
    ax=axes[1],
    legend=False,
    color="#b45f3c",
    title="Test MRR@20 (GAT-SAGPool)",
)
axes[1].set_ylabel("%")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
figure_path = RESULTS_DIR / f"{RESULT_FILE_PREFIX}_metrics.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
print(f"saved {figure_path}")

saved /kaggle/working/results/gat_sagpool_v2_digi_reg_metrics.png


## Cross-model integration

Two outputs are produced here:

1. `results/gat_sagpool_model_summary_rows.csv` - a self-contained CSV with the same schema as `results/model_summary.csv` and a `model="GAT-SAGPool"` column added. This is always written and is the file you would paste into the central comparison table during an offline merge.
2. If `results/model_summary.csv` already exists in `RESULTS_DIR` (e.g. uploaded as Kaggle input alongside this run), it is rewritten in place: any prior `GAT-SAGPool` rows are dropped, and the fresh ones are appended. Other models' rows are never touched. If no prior file exists, the SAGPool rows bootstrap a new `model_summary.csv`.

In addition, when both `results/gat_sr_gnn_results.csv` and `results/gat_tagnn_results.csv` happen to be present in `RESULTS_DIR`, a 3-way head-to-head comparison is written to `results/gat_three_way_comparison.csv` so SAGPool can be inspected next to the other two GAT branches without disturbing the existing 2-way `gat_head_to_head.csv`.

In [21]:
SAGPOOL_MODEL_LABEL = "GAT-SAGPool"

model_summary_additions = pd.DataFrame(
    [{"model": SAGPOOL_MODEL_LABEL, **row} for row in all_results]
)
model_summary_additions_path = (
    RESULTS_DIR / f"{RESULT_FILE_PREFIX}_model_summary_rows.csv"
)
model_summary_additions.to_csv(model_summary_additions_path, index=False)
print(f"saved {model_summary_additions_path}")

existing_summary_path = RESULTS_DIR / "model_summary.csv"
if existing_summary_path.exists():
    existing_summary = pd.read_csv(existing_summary_path)
    if "model" not in existing_summary.columns:
        print(
            f"existing summary at {existing_summary_path} has no 'model' column; "
            "skipping merge to avoid corrupting the file"
        )
    else:
        kept = existing_summary[existing_summary["model"] != SAGPOOL_MODEL_LABEL]
        merged = pd.concat([kept, model_summary_additions], ignore_index=True)
        merged.to_csv(existing_summary_path, index=False)
        print(f"updated {existing_summary_path}")
else:
    model_summary_additions.to_csv(existing_summary_path, index=False)
    print(f"created {existing_summary_path}")

sr_gnn_results_path = RESULTS_DIR / "gat_sr_gnn_results.csv"
tagnn_results_path = RESULTS_DIR / "gat_tagnn_results.csv"
if sr_gnn_results_path.exists() and tagnn_results_path.exists():
    sr_gnn_df = pd.read_csv(sr_gnn_results_path).set_index("dataset")
    tagnn_df = pd.read_csv(tagnn_results_path).set_index("dataset")
    sagpool_df = pd.DataFrame(all_results).set_index("dataset")

    head_to_head_rows = []
    for dataset_name in sagpool_df.index:
        if dataset_name not in sr_gnn_df.index or dataset_name not in tagnn_df.index:
            continue
        head_to_head_rows.append(
            {
                "dataset": dataset_name,
                "gat_sr_gnn_precision@20": sr_gnn_df.loc[
                    dataset_name, "test_precision@20"
                ],
                "gat_tagnn_precision@20": tagnn_df.loc[
                    dataset_name, "test_precision@20"
                ],
                "gat_sagpool_precision@20": sagpool_df.loc[
                    dataset_name, "test_precision@20"
                ],
                "gat_sr_gnn_mrr@20": sr_gnn_df.loc[dataset_name, "test_mrr@20"],
                "gat_tagnn_mrr@20": tagnn_df.loc[dataset_name, "test_mrr@20"],
                "gat_sagpool_mrr@20": sagpool_df.loc[dataset_name, "test_mrr@20"],
            }
        )

    if head_to_head_rows:
        head_to_head = pd.DataFrame(head_to_head_rows)
        head_to_head_path = RESULTS_DIR / "gat_three_way_comparison.csv"
        head_to_head.to_csv(head_to_head_path, index=False)
        try:
            display(head_to_head)
        except NameError:
            print(head_to_head.to_string(index=False))
        print(f"saved {head_to_head_path}")
    else:
        print("no overlapping datasets for 3-way comparison; skipping")
else:
    missing = [
        str(path)
        for path in (sr_gnn_results_path, tagnn_results_path)
        if not path.exists()
    ]
    print(f"3-way comparison skipped; missing baselines: {missing}")

saved /kaggle/working/results/gat_sagpool_v2_digi_reg_model_summary_rows.csv
created /kaggle/working/results/model_summary.csv
3-way comparison skipped; missing baselines: ['/kaggle/working/results/gat_sr_gnn_results.csv', '/kaggle/working/results/gat_tagnn_results.csv']


## Best-checkpoint upload (optional)

This mirrors the final cell of `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`: the best checkpoint (chosen by validation `MRR@20`) is published to KaggleHub under a SAGPool-specific slug. The call is wrapped so that an authentication failure or running outside Kaggle does not break the notebook flow - in that case the checkpoint stays available locally under `RESULTS_DIR / "checkpoints"`.

In [22]:
best_result = results.sort_values(
    ["best_validation_mrr@20", "best_validation_precision@20"],
    ascending=False,
).iloc[0]
best_dataset_slug = (
    best_result["dataset"].lower().replace(" ", "-").replace("/", "-")
)
best_checkpoint_path = Path(best_result["checkpoint_path"])

MODEL_SLUG = "gat-sagpool-session-recommender"
VARIATION_SLUG = f"best-{best_dataset_slug}"

try:
    kagglehub.model_upload(
        handle=f"karolbystrek/{MODEL_SLUG}/pytorch/{VARIATION_SLUG}",
        local_model_dir=str(best_checkpoint_path.parent),
        version_notes=(
            "Best GAT-SAGPool checkpoint selected by validation MRR@20. "
            f"Update {date.today().isoformat()}"
        ),
    )
    print(f"uploaded {MODEL_SLUG}/{VARIATION_SLUG} from {best_checkpoint_path.parent}")
except Exception as exc:
    print(f"kagglehub upload skipped ({exc.__class__.__name__}): {exc}")

Uploading Model https://api.kaggle.com/models/karolbystrek/gat-sagpool-session-recommender/pytorch/best-yoochoose-1-64 ...
Model 'gat-sagpool-session-recommender' does not exist or access is forbidden for user 'karolbystrek'. Creating or handling Model...
kagglehub upload skipped (BackendError): Permission 'models.create' was denied


## Phase 2 - Workstream B Architecture Sweep

This section runs the architecture sweep defined in `PLAN_SAGPOOL_IMPROVEMENT.md` Phase 2 - the eight Workstream B variants on the cheap `Yoochoose 1/64` testbed - in a single Kaggle session. The sweep is **enabled by default** (`RUN_PHASE2_B_SWEEP = True`); flip the flag to `False` to skip it. The baseline training cell above is independent and keeps running as before; on a single Kaggle commit the full pipeline (baseline + Phase 2 + Phase 3) takes ~`8h`, well within the 12h Kaggle budget.

For each variant the runner:

- starts from the active `config` (legacy `step3_g01` schedule, same baseline as the main training cell) and applies a single override pair so the effect of each architectural axis is attributable;
- runs `run_dataset_experiment` with a unique `result_file_prefix = f"phase2b_{label}"`, which means per-variant `*_history.csv`, `*_results.csv`, and per-variant `checkpoints/phase2b_{label}_<dataset>.pt` files;
- writes one summary row to `results/phase2b_sweep_summary.csv` after every variant completes (so partial sweeps are still useful if Kaggle times out).

Notes on the variant definitions:

- `b2_heads8_h128` combines B2 (`num_heads=8`) with B6 (`hidden_dim=128`) because `BidirectionalGATEncoder` requires `hidden_dim % num_heads == 0`. The "pure B6" variant `b6_hidden128` is also run, so the marginal effect of going from 4 to 8 heads can be isolated by comparing the two.
- `b3_blocks2` (hierarchical SAGPool) is the highest-risk variant because session graphs are tiny (2-6 unique items) and a second pool can collapse them. The smoke test catches the obvious failure modes; the sweep wraps each variant in `try/except` so one failing variant does not kill the rest.
- All variants inherit `epochs=30`, `patience=5` from the active config, so any single variant should finish on Yoochoose in ~22 minutes (best epoch is typically reached at epoch 6-8 and early stopping kicks in around epoch 12-14).

To validate the top picks on `Diginetica`, append `"Diginetica"` to `PHASE2_B_SWEEP_DATASETS` (each Diginetica run adds ~31 minutes per variant).

In [23]:
RUN_PHASE2_B_SWEEP = False                        # Run 1: sweeps off (STATUS-PRAC.md)
PHASE2_B_SWEEP_DATASETS = ["Yoochoose 1/64"]      # add "Diginetica" to validate top picks there

# B2 (num_heads=8) needs hidden_dim divisible by 8, so it is paired with hidden_dim=128.
# This conflates B2 with B6. The pure B6 variant ("b6_hidden128") is also in the list, so
# the marginal effect of going from 4 to 8 heads is recoverable from the comparison.
SWEEP_VARIANTS = [
    {"label": "b1_layers2",     "overrides": {"num_layers": 2}},
    {"label": "b2_heads8_h128", "overrides": {"num_heads": 8, "hidden_dim": 128}},
    {"label": "b3_blocks2",     "overrides": {"sagpool_blocks": 2}},
    {"label": "b4_ratio07",     "overrides": {"sagpool_ratio": 0.7}},
    {"label": "b4_ratio04",     "overrides": {"sagpool_ratio": 0.4}},
    {"label": "b5_gat_scorer",  "overrides": {"sagpool_scorer": "gat"}},
    {"label": "b6_hidden128",   "overrides": {"hidden_dim": 128}},
    {"label": "b7_gated",       "overrides": {"readout_combine": "gated"}},
]

if RUN_PHASE2_B_SWEEP:
    sweep_summary_path = RESULTS_DIR / "phase2b_sweep_summary.csv"
    sweep_summary_rows = []
    print(f"=== Phase 2 (Workstream B) sweep on {PHASE2_B_SWEEP_DATASETS} ===")
    print(f"variants: {[v['label'] for v in SWEEP_VARIANTS]}")

    for variant in SWEEP_VARIANTS:
        label = variant["label"]
        overrides = variant["overrides"]
        sweep_config = {**config, **overrides}
        sweep_prefix = f"phase2b_{label}"

        for dataset_name in PHASE2_B_SWEEP_DATASETS:
            dataset = DATASETS[dataset_name]
            print(f"\n--- variant '{label}' on {dataset_name}: overrides={overrides} ---")
            try:
                result, _ = run_dataset_experiment(
                    dataset_name,
                    dataset,
                    sweep_config,
                    device,
                    gat_sagpool_factory,
                    result_file_prefix=sweep_prefix,
                )
                row = {
                    "label": label,
                    "dataset": dataset_name,
                    "overrides": str(overrides),
                    "best_epoch": result["best_epoch"],
                    "val_p@20": result["best_validation_precision@20"],
                    "val_mrr@20": result["best_validation_mrr@20"],
                    "test_p@20": result["test_precision@20"],
                    "test_mrr@20": result["test_mrr@20"],
                }
                print(
                    f"variant '{label}' done: "
                    f"val_MRR@20={row['val_mrr@20']:.2f} "
                    f"test_P@20={row['test_p@20']:.2f} "
                    f"test_MRR@20={row['test_mrr@20']:.2f}"
                )
            except Exception as exc:
                print(
                    f"!!! variant '{label}' on {dataset_name} FAILED: "
                    f"{type(exc).__name__}: {exc}"
                )
                row = {
                    "label": label,
                    "dataset": dataset_name,
                    "overrides": str(overrides),
                    "best_epoch": -1,
                    "val_p@20": float("nan"),
                    "val_mrr@20": float("nan"),
                    "test_p@20": float("nan"),
                    "test_mrr@20": float("nan"),
                }
            sweep_summary_rows.append(row)
            # Stream summary to disk after every variant so a Kaggle timeout
            # still leaves the partial sweep usable.
            pd.DataFrame(sweep_summary_rows).to_csv(sweep_summary_path, index=False)

    print("\n=== Phase 2 (Workstream B) Sweep Summary ===")
    sweep_df = pd.DataFrame(sweep_summary_rows)
    sweep_df = sweep_df.sort_values(
        ["dataset", "val_mrr@20"], ascending=[True, False]
    )
    try:
        display(sweep_df)
    except NameError:
        print(sweep_df.to_string(index=False))
    print(f"saved {sweep_summary_path}")
else:
    print("Phase 2 (Workstream B) sweep is OFF (set RUN_PHASE2_B_SWEEP=True to enable it).")
    print(f"variants ready: {[v['label'] for v in SWEEP_VARIANTS]}")

Phase 2 (Workstream B) sweep is OFF (set RUN_PHASE2_B_SWEEP=True to enable it).
variants ready: ['b1_layers2', 'b2_heads8_h128', 'b3_blocks2', 'b4_ratio07', 'b4_ratio04', 'b5_gat_scorer', 'b6_hidden128', 'b7_gated']


## Phase 3 - Workstream C Regularization Sweep

This section runs the regularization sweep from `PLAN_SAGPOOL_IMPROVEMENT.md` Phase 3 - the seven Workstream C variants - on `Diginetica`, the dataset that the diagnostic analysis flagged as overfit-prone (`val_MRR@20 19.49 → test_MRR@20 16.74`, a `~2.75` point gap). The sweep is **enabled by default** (`RUN_PHASE3_C_SWEEP = True`); flip the flag to `False` to skip it.

For each variant the runner:

- starts from `PHASE3_BASE_OVERRIDES` (a dict of architectural overrides, empty by default). Once Phase 2 picks a B-winner, populate this dict (e.g. `{"sagpool_ratio": 0.7}`) so each C variant is evaluated on top of the chosen architecture.
- applies the C override on top, runs `run_dataset_experiment` with a unique `result_file_prefix = f"phase3c_{label}"`, and captures per-variant artifacts (history CSV, results CSV, dedicated checkpoint).
- writes one summary row to `results/phase3c_sweep_summary.csv` after every variant completes (so a partial sweep is still usable).

The sweep also reports the **val/test gap** (`val_mrr@20 - test_mrr@20`), since the explicit goal of Workstream C is to close this gap on Diginetica without sacrificing too much validation performance.

Notes on the variant definitions:

- `c1_dropout02`, `c1_dropout03` are the cheapest defenses; they tweak only an existing config knob.
- `c3_wd1e4` is similarly cheap and complementary to dropout.
- `c4_ls01` (label smoothing) is graph-orthogonal: it touches only the loss, leaving forward semantics intact.
- `c5_edge02` (edge dropout) and `c2_emb01` (embedding dropout) are graph- and item-aware regularizers - the most likely to help if the overfitting signal is in the encoder rather than the optimizer.
- `c_combo_best` stacks the typically-additive trio (dropout + weight_decay + label_smoothing). Edit the override dict if Phase 2 picks a different B-winner so the combo runs on top of the chosen architecture.

In [24]:
RUN_PHASE3_C_SWEEP = False                          # Run 1: sweeps off (STATUS-PRAC.md)
PHASE3_C_SWEEP_DATASETS = ["Diginetica"]            # add "Yoochoose 1/64" to also test there

# Architecture base for the C sweep. Empty by default = pure-baseline regularization
# screen. After Phase 2 picks a B-winner, populate this dict to evaluate each C
# variant on top of the chosen architecture (the plan's Phase 3 protocol).
PHASE3_BASE_OVERRIDES = {}

PHASE3_C_VARIANTS = [
    {"label": "c1_dropout02",  "overrides": {"dropout": 0.2}},
    {"label": "c1_dropout03",  "overrides": {"dropout": 0.3}},
    {"label": "c3_wd1e4",      "overrides": {"weight_decay": 1e-4}},
    {"label": "c4_ls01",       "overrides": {"label_smoothing": 0.1}},
    {"label": "c5_edge02",     "overrides": {"edge_dropout": 0.2}},
    {"label": "c2_emb01",      "overrides": {"embedding_dropout": 0.1}},
    # Stacks the orthogonal regularizers most often used together. Edit if a
    # different combination wins individually above.
    {
        "label": "c_combo_best",
        "overrides": {
            "dropout": 0.2,
            "weight_decay": 1e-4,
            "label_smoothing": 0.1,
            "embedding_dropout": 0.1,
        },
    },
]

if RUN_PHASE3_C_SWEEP:
    sweep_summary_path = RESULTS_DIR / "phase3c_sweep_summary.csv"
    sweep_summary_rows = []
    print(f"=== Phase 3 (Workstream C) sweep on {PHASE3_C_SWEEP_DATASETS} ===")
    print(f"base overrides: {PHASE3_BASE_OVERRIDES}")
    print(f"variants: {[v['label'] for v in PHASE3_C_VARIANTS]}")

    for variant in PHASE3_C_VARIANTS:
        label = variant["label"]
        c_overrides = variant["overrides"]
        sweep_config = {**config, **PHASE3_BASE_OVERRIDES, **c_overrides}
        sweep_prefix = f"phase3c_{label}"
        merged_overrides = {**PHASE3_BASE_OVERRIDES, **c_overrides}

        for dataset_name in PHASE3_C_SWEEP_DATASETS:
            dataset = DATASETS[dataset_name]
            print(
                f"\n--- variant '{label}' on {dataset_name}: "
                f"overrides={c_overrides} (over base {PHASE3_BASE_OVERRIDES}) ---"
            )
            try:
                result, _ = run_dataset_experiment(
                    dataset_name,
                    dataset,
                    sweep_config,
                    device,
                    gat_sagpool_factory,
                    result_file_prefix=sweep_prefix,
                )
                val_mrr = result["best_validation_mrr@20"]
                test_mrr = result["test_mrr@20"]
                row = {
                    "label": label,
                    "dataset": dataset_name,
                    "overrides": str(merged_overrides),
                    "best_epoch": result["best_epoch"],
                    "val_p@20": result["best_validation_precision@20"],
                    "val_mrr@20": val_mrr,
                    "test_p@20": result["test_precision@20"],
                    "test_mrr@20": test_mrr,
                    # Gap is what Workstream C is trying to close on Diginetica.
                    "val_minus_test_mrr@20": val_mrr - test_mrr,
                }
                print(
                    f"variant '{label}' done: val_MRR@20={val_mrr:.2f} "
                    f"test_MRR@20={test_mrr:.2f} gap={row['val_minus_test_mrr@20']:.2f}"
                )
            except Exception as exc:
                print(
                    f"!!! variant '{label}' on {dataset_name} FAILED: "
                    f"{type(exc).__name__}: {exc}"
                )
                row = {
                    "label": label,
                    "dataset": dataset_name,
                    "overrides": str(merged_overrides),
                    "best_epoch": -1,
                    "val_p@20": float("nan"),
                    "val_mrr@20": float("nan"),
                    "test_p@20": float("nan"),
                    "test_mrr@20": float("nan"),
                    "val_minus_test_mrr@20": float("nan"),
                }
            sweep_summary_rows.append(row)
            pd.DataFrame(sweep_summary_rows).to_csv(sweep_summary_path, index=False)

    print("\n=== Phase 3 (Workstream C) Sweep Summary ===")
    sweep_df = pd.DataFrame(sweep_summary_rows)
    sweep_df = sweep_df.sort_values(
        ["dataset", "test_mrr@20"], ascending=[True, False]
    )
    try:
        display(sweep_df)
    except NameError:
        print(sweep_df.to_string(index=False))
    print(f"saved {sweep_summary_path}")
else:
    print("Phase 3 (Workstream C) sweep is OFF (set RUN_PHASE3_C_SWEEP=True to enable it).")
    print(f"variants ready: {[v['label'] for v in PHASE3_C_VARIANTS]}")

Phase 3 (Workstream C) sweep is OFF (set RUN_PHASE3_C_SWEEP=True to enable it).
variants ready: ['c1_dropout02', 'c1_dropout03', 'c3_wd1e4', 'c4_ls01', 'c5_edge02', 'c2_emb01', 'c_combo_best']
